# Gemini: отдельное подключение через SOCKS-прокси

In [ ]:
from pathlib import Path
import sys, random
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'captcha_lab').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
random.seed(42)
import numpy as np
import pandas as pd
np.random.seed(42)
from captcha_lab.data import image_manifest, ensure_labels, crop_image, save_json
from captcha_lab.benchmark import run_benchmark, save_run, evaluate
print('Python:', sys.version.split()[0])

Секреты берутся из переменных среды или `.env.local` в корне проекта. Заполненный файл остаётся
только локально. Не вставляйте ключ в ячейки. Сначала выполняется один короткий текстовый запрос.
При неудаче распознавание изображений не запускается. Затем максимум 9 изображений, по одному запросу.
Повторный запуск делает новые платные/квотируемые запросы. Прямого fallback при ошибке прокси нет.
Используется заданный alias; фактическая версия ответа записывается отдельно.

In [ ]:
from captcha_lab.gemini import GeminiClient, GeminiError
client = GeminiClient.from_env()
print(repr(client))  # repr скрывает ключ и адрес/пароль прокси
RUN_IMAGE_TEST = True
MAX_IMAGES = 9

In [ ]:
smoke_ok = False
try:
    smoke = client.generate('Reply with only OK.')
    smoke_ok = smoke['text'] == 'OK'
    smoke['status'] = 'ok' if smoke_ok else 'unexpected_text'
except GeminiError as exc:
    smoke = {'status':'failed','error':str(exc)}
save_json(ROOT/'results/gemini-smoke.json',smoke)
display(smoke)

In [ ]:
if smoke_ok and RUN_IMAGE_TEST:
    manifest = image_manifest().head(MAX_IMAGES)
    assert not manifest.empty
    frame = run_benchmark(manifest,client.recognize,'gemini/'+client.model,repeats=1,warmup=False)
    frame.attrs['actual_model_version'] = smoke.get('model_version')
    save_run(frame,'gemini.csv')
    display(evaluate(frame,ensure_labels(manifest)))
else:
    print('Распознавание не запускалось: проверьте smoke и RUN_IMAGE_TEST')

Если smoke сообщает 401/403 — проверить ключ и доступ проекта; 429 — квоту; ошибку сети — прокси.
Это разные причины. Исключения намеренно не печатают исходные HTTP-ответы и секреты.
Отказ модели или лишний текст считаются ошибкой/отказом, а не очищаются до правдоподобного ответа.